# 第8章 回折

書籍『Sionna RTによる電波伝搬レイトレーシング入門』第8章のコードである。
セルは本文の掲載順に並べてあり、コードセルの中身は本文に印刷したものと同じである。

先頭の「準備」セルだけは本文に印刷していない。本文のコードが前提にしている
import とシーンの読み込みをここで済ませてある。準備セルを実行すれば、以降は
上から順に実行できる。

ただし、本文が説明のために示している断片（自分で作ったシーンのパスや、その
シーンには無い物体名を使う例）はそのままでは動かない。該当するセルの前に注記を
置いてある。

## 準備

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

# 本書の多くの章で使うミュンヘン中心部の都市シーン
scene = load_scene(scene_assets.munich)
scene.frequency = 3.5e9
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.add(Transmitter('tx0', position=[-30.0, 38.0, 35.0]))

solver = PathSolver()
rmsolver = RadioMapSolver()

## 4. Sionna RT で回折を有効にする

In [ ]:
import numpy as np

paths_no_diff = solver(scene, max_depth=3,
                       los=True, specular_reflection=True,
                       diffraction=False)
paths_diff = solver(scene, max_depth=3,
                    los=True, specular_reflection=True,
                    diffraction=True)
# a は (実部, 虚部) のタプルで返るので実部側で本数を数える
print('no diff:', np.array(paths_no_diff.a[0]).shape[-1])
print('with diff:', np.array(paths_diff.a[0]).shape[-1])

## 5. 回折の有無による比較

In [ ]:
rm_no = rmsolver(scene, max_depth=3, diffraction=False,
                 edge_diffraction=False,
                 cell_size=[2.0,2.0], samples_per_tx=10**6)
rm_with = rmsolver(scene, max_depth=3, diffraction=True,
                   edge_diffraction=True,
                   cell_size=[2.0,2.0], samples_per_tx=10**6)

## 7. 周波数と回折の関係

In [ ]:
for fc in [0.9e9, 3.5e9, 28e9]:
    scene.frequency = fc
    rm = rmsolver(scene, max_depth=3, diffraction=True,
                  edge_diffraction=True,
                  cell_size=[2.0,2.0], samples_per_tx=10**6)